#  KMUT Scheme — Synthetic Dataset Generator
## Kalaignar Magalir Urimai Thittam | Tamil Nadu Women's Rights Scheme

This notebook generates a **50,000-record synthetic dataset** simulating applicant data for the KMUT scheme.

### What it covers:
- Tamil Nadu district-level demographics
- Socioeconomic indicators (income, land, electricity)
- Eligibility flag computation (per official scheme rules)
- Application outcomes (Approved / Rejected / Pending)
- Financial benefit disbursement tracking
- Digital access & grievance data

##  1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, date, timedelta
import random
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
random.seed(42)

N = 50000  # Total records to generate
print(f"Will generate {N:,} records")

## 2. Reference Lists (Districts, Occupations, etc.)

In [ ]:
# Tamil Nadu districts (all 35)
DISTRICTS = [
    "Chennai", "Coimbatore", "Madurai", "Tiruchirappalli", "Salem",
    "Tirunelveli", "Vellore", "Erode", "Thoothukudi", "Dindigul",
    "Thanjavur", "Ranipet", "Namakkal", "Karur", "Kancheepuram",
    "Krishnagiri", "Dharmapuri", "Perambalur", "Ariyalur", "Villupuram",
    "Cuddalore", "Nagapattinam", "Tiruvarur", "Pudukkottai", "Sivaganga",
    "Virudhunagar", "Ramanathapuram", "Theni", "Nilgiris", "Tiruppur",
    "Tirupattur", "Kallakurichi", "Tenkasi", "Chengalpattu", "Mayiladuthurai"
]

URBAN_DISTRICTS = [
    "Chennai", "Coimbatore", "Madurai", "Tiruchirappalli", "Salem",
    "Tirunelveli", "Vellore", "Erode", "Thoothukudi", "Tiruppur",
    "Chengalpattu", "Kancheepuram"
]

OCCUPATIONS = [
    "Agricultural Laborer", "Daily Wage Worker", "Domestic Worker",
    "Street Vendor", "Tailoring/Weaving", "Self-employed (Small Business)",
    "NREGA Worker", "Homemaker", "Handloom Worker", "Fishery",
    "Construction Worker", "Beedi Worker", "Shop Assistant",
    "Sanitation Worker", "Anganwadi Helper", "Mid-Day Meal Worker"
]

EDUCATION_LEVELS = [
    "Illiterate", "Primary (1-5)", "Middle (6-8)", "Secondary (9-10)",
    "Higher Secondary (11-12)", "Diploma", "Graduate", "Post-Graduate"
]

MARITAL_STATUS = ["Married", "Widowed", "Divorced/Separated", "Single/Unmarried", "Transgender"]

BANK_NAMES = [
    "State Bank of India", "Indian Bank", "Canara Bank", "Indian Overseas Bank",
    "Bank of Baroda", "Tamil Nadu Mercantile Bank", "City Union Bank",
    "Karur Vysya Bank", "South Indian Bank", "UCO Bank"
]

REJECTION_REASONS = [
    "Income exceeds 2.5L", "Government employee in family",
    "Four-wheeler owned", "GST registered business",
    "Already receiving pension", "Duplicate application",
    "Ration card issue", "Bank account mismatch",
    "Age below 21", "Income tax filer in family"
]

print("Reference lists loaded:")
print(f"  Districts: {len(DISTRICTS)}")
print(f"  Occupations: {len(OCCUPATIONS)}")
print(f"  Education levels: {len(EDUCATION_LEVELS)}")

##  3. Helper Functions

In [ ]:
def random_date(start_year=2023, end_year=2025):
    """Generate a random date in the given year range."""
    start = date(start_year, 1, 1)
    end = date(end_year, 12, 31)
    delta = end - start
    return start + timedelta(days=random.randint(0, delta.days))

def generate_ration_card():
    """Generate a synthetic Tamil Nadu ration card number."""
    prefixes = ["TN", "MH", "CH", "CB", "MD"]
    return f"{random.choice(prefixes)}{random.randint(1000000, 9999999)}"

def generate_aadhaar():
    """Generate a masked Aadhaar number (last 4 digits will be used)."""
    return f"{random.randint(2000,9999)}-{random.randint(1000,9999)}-{random.randint(1000,9999)}"

def generate_bank_account():
    """Generate a synthetic bank account number."""
    return f"{random.randint(10000000000, 99999999999)}"

print("Helper functions defined!")

##  4. Generate Core Features

In [ ]:
print("Step 1: District & Area Type...")
raw_p = [0.12,0.08,0.07,0.06,0.05,0.04,0.04,0.03,0.03,0.03,
    0.03,0.03,0.03,0.02,0.03,0.02,0.02,0.01,0.01,0.02,
    0.02,0.02,0.02,0.02,0.02,0.02,0.02,0.02,0.01,0.03,
    0.01,0.01,0.01,0.02,0.01]
raw_p = [x/sum(raw_p) for x in raw_p]
district = np.random.choice(DISTRICTS, N, p=raw_p)

is_urban = np.array([d in URBAN_DISTRICTS for d in district])
area_type = np.where(is_urban,
    np.random.choice(["Urban", "Semi-Urban"], N, p=[0.7, 0.3]),
    np.random.choice(["Rural", "Semi-Rural"], N, p=[0.75, 0.25])
)

print("Step 2: Demographics...")
age = np.where(
    is_urban,
    np.random.randint(21, 75, N),
    np.random.randint(21, 70, N)
)

marital_probs = [0.55, 0.15, 0.10, 0.15, 0.05]
marital_status = np.random.choice(MARITAL_STATUS, N, p=marital_probs)
family_size = np.clip(np.random.normal(4.2, 1.5, N).astype(int), 1, 12)
social_category = np.random.choice(["SC","ST","OBC","General","MBC"], N, p=[0.20,0.05,0.30,0.15,0.30])

edu_probs = [0.08, 0.15, 0.18, 0.25, 0.15, 0.08, 0.09, 0.02]
education = np.random.choice(EDUCATION_LEVELS, N, p=edu_probs)

occupation_probs = [0.15,0.12,0.10,0.08,0.07,0.06,0.07,0.10,0.05,0.04,0.06,0.03,0.03,0.02,0.01,0.01]
occupation = np.random.choice(OCCUPATIONS, N, p=occupation_probs)

print("Step 3: Socioeconomic Indicators...")
base_income = np.exp(np.random.normal(10.5, 0.8, N))
annual_income = np.clip(base_income, 15000, 500000).astype(int)

has_land_flag = np.random.choice([True, False], N, p=[0.35, 0.65])
wet_land = np.where(has_land_flag, np.random.exponential(1.5, N), 0).round(2)
dry_land = np.where(has_land_flag, np.random.exponential(2.5, N), 0).round(2)
wet_land = np.clip(wet_land, 0, 15)
dry_land = np.clip(dry_land, 0, 25)

electricity_units = np.clip(np.random.normal(2200, 900, N).astype(int), 100, 8000)

print("Done generating core features!")
print(f"  Age range: {age.min()} - {age.max()}")
print(f"  Income range: Rs. {annual_income.min():,} - Rs. {annual_income.max():,}")

##  5. Compute Eligibility Flags (Per Scheme Rules)

In [ ]:
print("Computing eligibility per KMUT scheme rules...")

# Disqualifying conditions
has_income_tax    = (annual_income > 250000) & (np.random.random(N) < 0.6)
has_govt_employee = np.random.choice([True, False], N, p=[0.06, 0.94])
has_four_wheeler  = (annual_income > 300000) & (np.random.random(N) < 0.4)
has_gst_business  = (annual_income > 400000) & (np.random.random(N) < 0.3)
already_has_pension = np.random.choice([True, False], N, p=[0.08, 0.92])

# Economic eligibility criteria
income_eligible      = annual_income <= 250000
land_eligible        = (wet_land < 5) & (dry_land < 10)
electricity_eligible = electricity_units <= 3600
economically_eligible = income_eligible & land_eligible & electricity_eligible

# Combined ineligibility
ineligible_flag = (has_income_tax | has_govt_employee | has_four_wheeler |
                   has_gst_business | already_has_pension)

# Final eligibility
final_eligible = economically_eligible & ~ineligible_flag

print(f"Eligibility breakdown:")
print(f"  Income eligible:       {income_eligible.sum():,} ({income_eligible.mean()*100:.1f}%)")
print(f"  Land eligible:         {land_eligible.sum():,} ({land_eligible.mean()*100:.1f}%)")
print(f"  Electricity eligible:  {electricity_eligible.sum():,} ({electricity_eligible.mean()*100:.1f}%)")
print(f"  Economically eligible: {economically_eligible.sum():,} ({economically_eligible.mean()*100:.1f}%)")
print(f"  FINAL ELIGIBLE:        {final_eligible.sum():,} ({final_eligible.mean()*100:.1f}%)")

##  6. Generate Application Outcomes

In [ ]:
print("Generating application outcomes...")

# Application decision (eligible people more likely to apply)
applied = np.zeros(N, dtype=bool)
applied[final_eligible]  = np.random.choice([True, False], final_eligible.sum(),  p=[0.75, 0.25])
applied[~final_eligible] = np.random.choice([True, False], (~final_eligible).sum(), p=[0.15, 0.85])

# Status per application
app_status = np.full(N, "Not Applied", dtype=object)
for i in range(N):
    if applied[i]:
        if final_eligible[i]:
            app_status[i] = np.random.choice(
                ["Approved","Pending","Under Review","Rejected"], p=[0.70,0.12,0.10,0.08])
        else:
            app_status[i] = np.random.choice(["Rejected","Pending","Under Review"], p=[0.80,0.10,0.10])

# Financial outcomes
monthly_benefit        = np.where(app_status == "Approved", 1000, 0)
months_received        = np.where(app_status == "Approved", np.random.randint(1, 25, N), 0)
total_benefit_received = monthly_benefit * months_received

# Dates
app_date = [random_date(2023, 2025) if applied[i] else None for i in range(N)]
approval_date = []
for i in range(N):
    if app_status[i] == "Approved" and app_date[i]:
        approval_date.append(app_date[i] + timedelta(days=random.randint(15, 60)))
    else:
        approval_date.append(None)

import collections
status_counts = collections.Counter(app_status)
print(f"Application status distribution:")
for k, v in sorted(status_counts.items(), key=lambda x: -x[1]):
    print(f"  {k:<15} {v:>7,}")

##  7. Generate Digital Access & Grievance Data

In [ ]:
# Bank & digital access
bank_name         = np.random.choice(BANK_NAMES, N)
has_bank_account  = np.random.choice([True, False], N, p=[0.88, 0.12])
has_smartphone    = np.random.choice([True, False], N, p=[0.65, 0.35])
digital_literacy  = np.random.choice(["None","Basic","Intermediate","Advanced"], N, p=[0.35,0.40,0.18,0.07])
fps_distance_km   = np.clip(np.random.exponential(3.5, N), 0.1, 30).round(1)
hof_gender        = np.random.choice(["Male","Female"], N, p=[0.62, 0.38])
is_direct_hof     = (hof_gender == "Female")
is_bpl            = (annual_income < 100000) & (np.random.random(N) < 0.70)

# Grievances
grievance_filed    = np.where(
    (app_status == "Rejected") | (app_status == "Pending"),
    np.random.choice([True, False], N, p=[0.25, 0.75]), False)
grievance_resolved = np.where(
    grievance_filed, np.random.choice([True, False], N, p=[0.60, 0.40]), False)

# Satisfaction (approved only)
satisfaction_score = np.where(
    app_status == "Approved",
    np.random.choice([1,2,3,4,5], N, p=[0.03,0.07,0.15,0.40,0.35]),
    np.nan)

# Rejection reason
ration_card = [generate_ration_card() for _ in range(N)]
aadhaar     = [generate_aadhaar() for _ in range(N)]

rejection_reason = []
for i in range(N):
    if app_status[i] == "Rejected":
        if ineligible_flag[i]:
            if has_income_tax[i]:      rejection_reason.append("Income exceeds 2.5L / Tax filer")
            elif has_govt_employee[i]: rejection_reason.append("Government employee in family")
            elif has_four_wheeler[i]:  rejection_reason.append("Four-wheeler owned")
            elif has_gst_business[i]:  rejection_reason.append("GST registered business")
            elif already_has_pension[i]: rejection_reason.append("Already receiving pension")
            else:                      rejection_reason.append(random.choice(REJECTION_REASONS))
        else:
            rejection_reason.append(random.choice(["Duplicate application","Bank account mismatch","Ration card issue"]))
    else:
        rejection_reason.append(None)

print(f"Digital access: {has_bank_account.mean()*100:.1f}% banked | {has_smartphone.mean()*100:.1f}% smartphone")
print(f"Grievances filed: {grievance_filed.sum():,} | Resolved: {grievance_resolved.sum():,}")

##  8. Assemble Final DataFrame

In [ ]:
applicant_id = [f"KMUT{str(i+1).zfill(6)}" for i in range(N)]

df = pd.DataFrame({
    "applicant_id": applicant_id,
    "district": district,
    "area_type": area_type,
    "age": age,
    "marital_status": marital_status,
    "social_category": social_category,
    "education_level": education,
    "occupation": occupation,
    "family_size": family_size,
    "annual_income_inr": annual_income,
    "wet_land_acres": wet_land,
    "dry_land_acres": dry_land,
    "electricity_units_year": electricity_units,
    "hof_gender": hof_gender,
    "is_direct_hof_female": is_direct_hof,
    "is_bpl": is_bpl,
    "has_bank_account": has_bank_account,
    "bank_name": bank_name,
    "has_smartphone": has_smartphone,
    "digital_literacy": digital_literacy,
    "fps_distance_km": fps_distance_km,
    "has_four_wheeler": has_four_wheeler,
    "has_govt_employee_family": has_govt_employee,
    "has_income_tax_filer": has_income_tax,
    "has_gst_business": has_gst_business,
    "already_receiving_pension": already_has_pension,
    "income_eligible": income_eligible,
    "land_eligible": land_eligible,
    "electricity_eligible": electricity_eligible,
    "economically_eligible": economically_eligible,
    "final_eligible": final_eligible,
    "applied_for_scheme": applied,
    "application_status": app_status,
    "application_date": app_date,
    "approval_date": approval_date,
    "monthly_benefit_inr": monthly_benefit,
    "months_benefit_received": months_received,
    "total_benefit_received_inr": total_benefit_received,
    "rejection_reason": rejection_reason,
    "grievance_filed": grievance_filed,
    "grievance_resolved": grievance_resolved,
    "satisfaction_score": satisfaction_score,
    "ration_card_number": ration_card,
    "aadhaar_last4": [a[-4:] for a in aadhaar],
})

print(f"DataFrame assembled: {df.shape}")
print(f"\nColumn dtypes:\n{df.dtypes.value_counts()}")
df.head(3)

##  9. Quick Summary Statistics

In [ ]:
print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)
print(f"Total Records:          {len(df):,}")
print(f"Total Columns:          {df.shape[1]}")
print(f"Eligible Households:    {df.final_eligible.sum():,} ({df.final_eligible.mean()*100:.1f}%)")
print(f"Applied:                {df.applied_for_scheme.sum():,}")
print(f"Approved:               {(df.application_status=='Approved').sum():,}")
print(f"Rejected:               {(df.application_status=='Rejected').sum():,}")
print(f"Total Benefit Disbursed:Rs. {df.total_benefit_received_inr.sum()/1e7:.2f} Crores")
print(f"Avg Satisfaction Score: {df.satisfaction_score.mean():.2f}/5")
print()
print(df[['age','annual_income_inr','family_size','electricity_units_year',
          'fps_distance_km','months_benefit_received']].describe().round(2))

##  10. Save Dataset

In [ ]:
output_path = "kmut_dataset.csv"
df.to_csv(output_path, index=False)
print(f"Dataset saved to: {output_path}")
print(f"File size: ~{df.memory_usage(deep=True).sum() / 1e6:.1f} MB in memory")
print("\n Dataset generation complete! Ready for Kaggle upload.")